<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/Phase45_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 45 — private Athena and independent binding revalidation

Phase 44 pinned the public OMOP CDM v5.5 and MI-CDM schema revisions. Phase 45 returns to the unresolved evidence gates that prevent an official Arm A transformation.

This notebook searches Google Drive for the existing Phase 44 artifacts and for the private vocabulary-review materials created by the earlier Phase 38 workflow. It never copies selected concept IDs, Athena records, reviewer identities, or private paths into the public Phase 45 outputs.

### Evidence evaluated
- authorized frozen `CONCEPT.csv` or `CONCEPT.csv.gz`
- matching `SNAPSHOT_MANIFEST.json`
- independently reviewed `ICHI2027_Phase38_REVIEWED_BINDINGS.csv`
- the original Phase 29 gap-disposition artifact, when available
- an independent 36-gap resolution ledger
- an independent missing-demographics policy review

If required private evidence is absent, the notebook creates private templates without overwriting existing work and leaves the corresponding gate blocked.

### Boundary
Passing vocabulary review does not create missing clinical Procedures or real DICOM UIDs. Phase 44 established that those are required by the pinned imaging schema, so official Arm A remains blocked until source-supported evidence exists.


In [1]:
# CODE CELL 1/8 — Drive, constants, and helpers
import os, re, csv, json, gzip, hashlib, shutil, tempfile, zipfile, time, datetime
from pathlib import Path
from collections import Counter

PROJECT='ICHI2027_neuroimaging_fhir_omop_fidelity'
PHASE='45_private_athena_binding_gap_and_demographic_policy_revalidation'

EXPECTED={
 'summary':('ICHI2027_Phase44_SHAREABLE','.json','a4f2aa36c472f82f630cdb102d2ea742f7f4bc6d9737d1f9d0d4fa9790b80747'),
 'lock':('ICHI2027_Phase44_STANDARDS_LOCK','.json','0e11c4909874c344ec778332c9eec4d52149270b4340d1b21c2096b5aa8d7a84'),
 'gates':('ICHI2027_Phase44_ARM_A_GATES_UNCHANGED','.csv','4c1febebd83ee969f9102499ddd09fba6542092c202fcfd7d98015797f1f454b'),
 'contract':('ICHI2027_Phase44_LOGICAL_CONTRACT_CARRIED_FORWARD','.csv','d80e7ede309fedca3e95341b299ae4756c9a6796ea2148c4abc0479f8ee235da'),
 'preflight':('ICHI2027_Phase44_SCHEMA_BINDING_PREFLIGHT','.csv','363257938ccea3a638e27822298d18df680cb68a456782c12bfb8854cadf3e40'),
 'findings':('ICHI2027_Phase44_SCHEMA_FINDINGS','.csv','00de10020f4a50ffecee1c8fdeeef7948de16422b07138cf49fffdc5c708dfe0'),
}

BINDING_KEYS=[
 'IO_MODALITY_MR','IF_HIP_LEFT','IF_HIP_RIGHT','IF_ENT_LEFT','IF_ENT_RIGHT',
 'IF_TYPE_ALGORITHM','PERSON_GENDER_F','PERSON_GENDER_M',
 'PERSON_RACE_UNRECORDED','PERSON_ETHNICITY_UNRECORDED','PROCEDURE_MRI','PROCEDURE_TYPE_SYN',
 'MEAS_HIP_LEFT','MEAS_HIP_RIGHT','MEAS_ENT_LEFT','MEAS_ENT_RIGHT',
 'MEAS_TYPE_SYN','UNIT_MM3','IF_EVENT_MEAS_ID'
]
EXPECTED_DOMAINS={k:'' for k in BINDING_KEYS}
EXPECTED_DOMAINS.update({
 'PERSON_GENDER_F':'Gender','PERSON_GENDER_M':'Gender',
 'PERSON_RACE_UNRECORDED':'Race','PERSON_ETHNICITY_UNRECORDED':'Ethnicity',
 'PROCEDURE_MRI':'Procedure','PROCEDURE_TYPE_SYN':'Type Concept',
 'MEAS_HIP_LEFT':'Measurement','MEAS_HIP_RIGHT':'Measurement',
 'MEAS_ENT_LEFT':'Measurement','MEAS_ENT_RIGHT':'Measurement',
 'MEAS_TYPE_SYN':'Type Concept','UNIT_MM3':'Unit','IF_EVENT_MEAS_ID':'Metadata'
})
REVIEW_COLS=[
 'binding_key','proposed_concept_id','snapshot_sha256','reviewer','review_date_utc',
 'independent_review_evidence','exact_source_code_and_definition','semantic_and_domain_justification',
 'source_application_policy','independent_review_decision','extension_domain_policy_evidence',
 'laterality_preservation_evidence','event_field_cdm_table_field','event_field_definition_evidence'
]

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2*1024*1024),b''):
            h.update(b)
    return h.hexdigest()

def parse_date(raw):
    s=str(raw or '').strip()
    for fmt in ('%Y-%m-%d','%Y%m%d'):
        try:return datetime.datetime.strptime(s,fmt).date()
        except ValueError:pass
    return None

def write_csv(path,rows,fieldnames=None):
    rows=list(rows)
    if fieldnames is None:
        if not rows: raise ValueError('fieldnames required for empty CSV')
        fieldnames=list(rows[0].keys())
    with Path(path).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fieldnames)
        w.writeheader(); w.writerows(rows)

if os.environ.get('PHASE45_LOCAL_DRIVE_ROOT'):
    MYDRIVE=Path(os.environ['PHASE45_LOCAL_DRIVE_ROOT']).expanduser().resolve()
    ROOTS=[MYDRIVE]
    print('Offline Drive-mirror mode:',MYDRIVE)
else:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    MYDRIVE=Path('/content/drive/MyDrive')
    assert MYDRIVE.is_dir()
    ROOTS=[MYDRIVE]
    sd=Path('/content/drive/Shareddrives')
    if sd.is_dir(): ROOTS.append(sd)

hint=os.environ.get('PHASE45_DRIVE_FOLDER_HINT','').strip()
if hint:
    hp=Path(hint).expanduser()
    if not hp.is_absolute(): hp=MYDRIVE/hp
    if not hp.is_dir(): raise FileNotFoundError(hp)
    ROOTS=[hp.resolve()]

WORK=Path(tempfile.mkdtemp(prefix='ichi2027_phase45_'))
STAGE=WORK/'outputs'; STAGE.mkdir()
OUT_BASE=Path(os.environ.get('PHASE45_OUTPUT_DIR',str(MYDRIVE/'ICHI2027'/'Phase45')))
OUT=OUT_BASE/datetime.datetime.now(datetime.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')
print('Phase 45 output:',OUT)


Mounted at /content/drive
Phase 45 output: /content/drive/MyDrive/ICHI2027/Phase45/run_20261001T020553_159729Z


In [2]:
# CODE CELL 2/8 — Find exact Phase 44 chain by SHA-256

def name_matches(actual,stem,ext):
    return re.fullmatch(re.escape(stem)+r'(?:-\d+|\s*\(\d+\))?'+re.escape(ext),actual,re.I) is not None

P44={}
for key,(stem,ext,digest) in EXPECTED.items():
    good=[]
    for root in ROOTS:
        for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
            dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
            for fn in files:
                if name_matches(fn,stem,ext):
                    p=Path(here)/fn
                    try:
                        if sha256(p)==digest: good.append(p)
                    except OSError: pass
    if not good:
        raise FileNotFoundError(f'No verified Drive copy found for {stem}{ext}')
    P44[key]=sorted(good,key=str)[0]

summary44=json.loads(P44['summary'].read_text('utf-8'))
lock44=json.loads(P44['lock'].read_text('utf-8'))
with P44['gates'].open(newline='',encoding='utf-8-sig') as f:gates44=list(csv.DictReader(f))
with P44['contract'].open(newline='',encoding='utf-8-sig') as f:contract44=list(csv.DictReader(f))
with P44['preflight'].open(newline='',encoding='utf-8-sig') as f:preflight44=list(csv.DictReader(f))
with P44['findings'].open(newline='',encoding='utf-8-sig') as f:findings44=list(csv.DictReader(f))

assert summary44['phase']=='44_public_standards_snapshot_and_armA_schema_binding_preflight'
assert summary44['omop_version']=='5.5'
assert summary44['micdm_declared_ddl_version']=='1.0'
assert summary44['armA_unapproved_original_gates']==10
assert summary44['approved_vocabulary_bindings']==0
assert len(gates44)==10 and len(contract44)==8 and len(preflight44)==8
assert summary44['image_occurrence_procedure_occurrence_id_required'] is True
assert summary44['image_occurrence_study_uid_required'] is True
assert summary44['image_occurrence_series_uid_required'] is True

print('PASS: exact Phase 44 standards/preflight chain verified.')
print('OMOP commit:',lock44['omop_repo_commit'])
print('ImageWG commit:',lock44['imagewg_repo_commit'])


PASS: exact Phase 44 standards/preflight chain verified.
OMOP commit: 4a910305b2cb74a4fc2b2c34baf44eb0542ff03f
ImageWG commit: 8f0a19bb94e941bb335a097bc9851fab672621c7


In [3]:
# CODE CELL 3/8 — Locate the private evidence folder; create non-destructive templates when absent

private_dirs=[]
for root in ROOTS:
    for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
        dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
        if Path(here).name=='_PRIVATE_ATHENA':
            private_dirs.append(Path(here))

def evidence_score(d):
    wanted=['ICHI2027_Phase38_REVIEWED_BINDINGS.csv','SNAPSHOT_MANIFEST.json','CONCEPT.csv','CONCEPT.csv.gz','concept.csv','concept.csv.gz']
    return sum((d/x).is_file() for x in wanted)

if private_dirs:
    PRIVATE=sorted(private_dirs,key=lambda p:(-evidence_score(p),str(p)))[0]
else:
    PRIVATE=MYDRIVE/'ICHI2027'/'_PRIVATE_ATHENA'
    PRIVATE.mkdir(parents=True,exist_ok=True)

BINDINGS=PRIVATE/'ICHI2027_Phase38_REVIEWED_BINDINGS.csv'
if not BINDINGS.exists():
    rows=[]
    for k in BINDING_KEYS:
        r={c:'' for c in REVIEW_COLS}
        r['binding_key']=k
        r['independent_review_decision']='PENDING'
        rows.append(r)
    write_csv(BINDINGS,rows,REVIEW_COLS)
    print('Created private binding-review template.')
else:
    print('Existing private binding-review worksheet found.')

MANIFEST_TEMPLATE=PRIVATE/'SNAPSHOT_MANIFEST_TEMPLATE.json'
if not MANIFEST_TEMPLATE.exists():
    MANIFEST_TEMPLATE.write_text(json.dumps({
      'snapshot_date_utc':'YYYY-MM-DD',
      'expected_CONCEPT_sha256':'PASTE_FULL_SHA256_OF_EXACT_CONCEPT_FILE',
      'source':'Authorized OHDSI Athena export origin and export date',
      'omop_vocabulary_release':'Exact release',
      'independence_note':'Keep signed independent-review evidence with this private folder'
    },indent=2)+'\n',encoding='utf-8')

DEMO_REVIEW=PRIVATE/'ICHI2027_Phase45_DEMOGRAPHIC_POLICY_REVIEW.csv'
DEMO_COLS=['field','verified_source_state','proposed_policy','reviewer','review_date_utc','independent_review_evidence','decision']
if not DEMO_REVIEW.exists():
    demo=[]
    for field in ['gender','birthDate','race','ethnicity']:
        demo.append({
          'field':field,
          'verified_source_state':'ABSENT_IN_VERIFIED_SYNTHETIC_SOURCE',
          'proposed_policy':'',
          'reviewer':'',
          'review_date_utc':'',
          'independent_review_evidence':'',
          'decision':'PENDING'
        })
    write_csv(DEMO_REVIEW,demo,DEMO_COLS)
    print('Created private demographic-policy review template.')

print('Private evidence workspace is ready. Its contents are excluded from public Phase 45 artifacts.')


Existing private binding-review worksheet found.
Private evidence workspace is ready. Its contents are excluded from public Phase 45 artifacts.


In [4]:
# CODE CELL 4/8 — Validate frozen Athena snapshot, manifest, and the 19 human binding decisions

ATHENA=next((p for p in [
    PRIVATE/'CONCEPT.csv',PRIVATE/'CONCEPT.csv.gz',PRIVATE/'concept.csv',PRIVATE/'concept.csv.gz'
] if p.is_file()),None)
SNAPSHOT_HASH=sha256(ATHENA) if ATHENA else None
MANIFEST=PRIVATE/'SNAPSHOT_MANIFEST.json'

manifest={}
snapshot_day=None
manifest_valid=False
if MANIFEST.is_file():
    try:
        manifest=json.loads(MANIFEST.read_text('utf-8'))
        snapshot_day=datetime.date.fromisoformat(str(manifest['snapshot_date_utc']))
        manifest_valid=(
          ATHENA is not None
          and snapshot_day<=datetime.datetime.now(datetime.timezone.utc).date()
          and bool(str(manifest.get('source','')).strip())
          and bool(str(manifest.get('omop_vocabulary_release','')).strip())
          and str(manifest.get('expected_CONCEPT_sha256','')).strip().lower()==SNAPSHOT_HASH
        )
    except (ValueError,TypeError,KeyError,json.JSONDecodeError):
        manifest_valid=False

binding_structure_valid=False
rv=[]
try:
    with BINDINGS.open(newline='',encoding='utf-8-sig') as f:rv=list(csv.DictReader(f))
    binding_structure_valid=(
      len(rv)==19
      and len({r.get('binding_key','') for r in rv})==19
      and {r.get('binding_key','') for r in rv}==set(BINDING_KEYS)
      and bool(rv)
      and set(REVIEW_COLS).issubset(rv[0].keys())
    )
except Exception:
    binding_structure_valid=False

if not binding_structure_valid:
    raise RuntimeError('Private binding worksheet exists but does not match the required 19-row Phase 38 structure. It was not overwritten.')

reviews={r['binding_key']:r for r in rv}
chosen={r['proposed_concept_id'].strip() for r in rv if r['proposed_concept_id'].strip()}
if not all(x.isdigit() and int(x)>0 for x in chosen):
    raise RuntimeError('The private worksheet contains a non-positive or placeholder concept ID.')

needed={'concept_id','concept_name','domain_id','standard_concept','valid_start_date','valid_end_date','invalid_reason'}
matching={}
athena_schema_valid=False
if ATHENA:
    opener=gzip.open if ATHENA.suffix.lower()=='.gz' else open
    with opener(ATHENA,'rt',encoding='utf-8-sig',errors='replace',newline='') as f:
        first=f.readline()
        sep='\t' if '\t' in first else ','
    with opener(ATHENA,'rt',encoding='utf-8-sig',errors='replace',newline='') as f:
        reader=csv.DictReader(f,delimiter=sep)
        fields=set(reader.fieldnames or [])
        athena_schema_valid=needed.issubset(fields)
        if not athena_schema_valid:
            raise RuntimeError('Private CONCEPT export is missing required OMOP vocabulary columns.')
        for c in reader:
            cid=str(c.get('concept_id','')).strip()
            if cid in chosen:
                matching[cid]={k:c.get(k,'') for k in needed}

status=[];approved_bindings=[];inapplicable=[]
today=datetime.datetime.now(datetime.timezone.utc).date()

for key in BINDING_KEYS:
    r=reviews[key]
    cid=r['proposed_concept_id'].strip()
    disp=r['independent_review_decision'].strip().upper()
    s='BLOCKED_PENDING_INDEPENDENT_REVIEW'
    documented=all(str(r[x]).strip() for x in [
      'reviewer','review_date_utc','independent_review_evidence',
      'exact_source_code_and_definition','semantic_and_domain_justification','source_application_policy'
    ])
    reviewer_day=parse_date(r['review_date_utc'])
    reviewer_date_ok=reviewer_day is not None and reviewer_day<=today

    if key in {'PERSON_GENDER_F','PERSON_GENDER_M'}:
        if cid or disp=='APPROVED':
            s='REJECTED_SOURCE_GENDER_ABSENT'
        elif disp=='NOT_APPLICABLE_SOURCE_ABSENT' and documented and reviewer_date_ok:
            s='REVIEWED_INAPPLICABLE_SOURCE_ABSENT'; inapplicable.append(key)
        elif disp!='PENDING':
            s='BLOCKED_GENDER_MUST_BE_MARKED_NOT_APPLICABLE'

    elif disp=='NOT_APPLICABLE_SOURCE_ABSENT':
        s='BLOCKED_MISSINGNESS_REQUIRES_EXPLICIT_PERSON_POLICY' if key.startswith('PERSON_') else 'BLOCKED_UNSUPPORTED_NOT_APPLICABLE'

    elif cid or disp=='APPROVED':
        if not cid or disp!='APPROVED':
            s='BLOCKED_ID_AND_APPROVAL_REQUIRED'
        elif ATHENA is None:
            s='BLOCKED_NO_PRIVATE_ATHENA_EXPORT'
        elif not manifest_valid:
            s='BLOCKED_SNAPSHOT_MANIFEST_MISSING_OR_MISMATCHED'
        elif r['snapshot_sha256'].strip().lower()!=SNAPSHOT_HASH:
            s='BLOCKED_REVIEW_SNAPSHOT_SHA_MISMATCH'
        elif not documented or not reviewer_date_ok:
            s='BLOCKED_HUMAN_REVIEW_EVIDENCE_OR_DATE_MISSING'
        elif cid not in matching:
            s='BLOCKED_CONCEPT_NOT_IN_FROZEN_EXPORT'
        else:
            c=matching[cid]
            start=parse_date(c['valid_start_date']); end=parse_date(c['valid_end_date'])
            expected=EXPECTED_DOMAINS[key]
            if c['invalid_reason'].strip() or not start or not end or not start<=snapshot_day<=end:
                s='BLOCKED_CONCEPT_EXPIRED_OR_INVALID'
            elif expected and c['domain_id']!=expected:
                s='BLOCKED_DOMAIN_MISMATCH'
            elif key!='IF_EVENT_MEAS_ID' and c['standard_concept']!='S':
                s='BLOCKED_NONSTANDARD_CONCEPT'
            elif not expected and not r['extension_domain_policy_evidence'].strip():
                s='BLOCKED_EXTENSION_DOMAIN_POLICY_UNDOCUMENTED'
            elif key in {'IF_HIP_LEFT','IF_HIP_RIGHT','IF_ENT_LEFT','IF_ENT_RIGHT'} and not r['laterality_preservation_evidence'].strip():
                s='BLOCKED_LATERALITY_NOT_JUSTIFIED'
            elif key=='IF_EVENT_MEAS_ID' and not (
                c['concept_name'].strip().lower()=='measurement_id'
                and r['event_field_cdm_table_field'].strip().upper()=='MEASUREMENT.MEASUREMENT_ID'
                and r['event_field_definition_evidence'].strip()
            ):
                s='BLOCKED_EVENT_FIELD_METADATA_NOT_ADJUDICATED'
            elif key=='PERSON_RACE_UNRECORDED' and 'unrecorded' not in r['source_application_policy'].lower():
                s='BLOCKED_RACE_MISSINGNESS_POLICY'
            elif key=='PERSON_ETHNICITY_UNRECORDED' and 'unrecorded' not in r['source_application_policy'].lower():
                s='BLOCKED_ETHNICITY_MISSINGNESS_POLICY'
            elif key in {'PROCEDURE_MRI','PROCEDURE_TYPE_SYN'} and 'synthetic' not in r['source_application_policy'].lower():
                s='BLOCKED_MANUFACTURED_PROCEDURE_POLICY'
            else:
                s='REVIEWED_SNAPSHOT_CHECKED_SYNTHETIC_ONLY'; approved_bindings.append(key)

    status.append({
      'binding_key':key,
      'status':s,
      'expected_domain_if_defined':EXPECTED_DOMAINS[key],
      'candidate_present':bool(cid),
      'manual_review_supplied':disp!='PENDING',
      'snapshot_present':ATHENA is not None,
      'manifest_valid':manifest_valid
    })

binding_gate=(len(approved_bindings)+len(inapplicable)==19)
print('Frozen Athena present:',ATHENA is not None)
print('Manifest valid:',manifest_valid)
print('Snapshot-checked approvals:',len(approved_bindings))
print('Reviewed source-absent gender keys:',len(inapplicable))
print('19-binding gate:',binding_gate)


Frozen Athena present: False
Manifest valid: False
Snapshot-checked approvals: 0
Reviewed source-absent gender keys: 0
19-binding gate: False


In [5]:
# CODE CELL 5/8 — Reconstruct the 36-gap evidence ledger when the original instance file exists

# Search prior public artifact by filename. We do not treat the Phase37 summary count as 36 independently reviewed instances.
phase29_candidates=[]
for root in ROOTS:
    for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
        dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
        for fn in files:
            if re.fullmatch(r'ICHI2027_Phase29_GAP_DISPOSITION(?:-\d+|\s*\(\d+\))?\.csv',fn,re.I):
                phase29_candidates.append(Path(here)/fn)

GAP_SOURCE=None
gap_source_rows=[]
gap_source_hashes=[]
if phase29_candidates:
    # Prefer a candidate with exactly 36 data rows.
    for p in sorted(phase29_candidates,key=str):
        try:
            with p.open(newline='',encoding='utf-8-sig') as f:
                rows=list(csv.DictReader(f))
            if len(rows)==36:
                GAP_SOURCE=p; gap_source_rows=rows; break
        except Exception:
            pass

if GAP_SOURCE:
    def canonical_row_hash(row):
        blob=json.dumps({k:str(v or '') for k,v in sorted(row.items())},sort_keys=True,separators=(',',':')).encode('utf-8')
        return hashlib.sha256(blob).hexdigest()
    gap_source_hashes=[canonical_row_hash(r) for r in gap_source_rows]
    assert len(gap_source_hashes)==36 and len(set(gap_source_hashes))==36

GAP_REVIEW=PRIVATE/'ICHI2027_Phase45_REVIEWED_GAP_RESOLUTION.csv'
GAP_COLS=[
 'gap_instance_id','source_row_sha256','binding_key','reviewer','review_date_utc',
 'independent_review_evidence','resolution_status','resolution_note'
]

if not GAP_REVIEW.exists():
    template=[]
    if GAP_SOURCE:
        for i,h in enumerate(gap_source_hashes,1):
            template.append({
              'gap_instance_id':f'GAP{i:03d}','source_row_sha256':h,'binding_key':'',
              'reviewer':'','review_date_utc':'','independent_review_evidence':'',
              'resolution_status':'PENDING','resolution_note':''
            })
    else:
        for i in range(1,37):
            template.append({
              'gap_instance_id':f'GAP{i:03d}','source_row_sha256':'','binding_key':'',
              'reviewer':'','review_date_utc':'','independent_review_evidence':'',
              'resolution_status':'PENDING','resolution_note':''
            })
    write_csv(GAP_REVIEW,template,GAP_COLS)
    print('Created private 36-gap review template.')

with GAP_REVIEW.open(newline='',encoding='utf-8-sig') as f:
    gap_reviews=list(csv.DictReader(f))

gap_structure_ok=(
  len(gap_reviews)==36
  and len({r.get('gap_instance_id','') for r in gap_reviews})==36
  and bool(gap_reviews)
  and set(GAP_COLS).issubset(gap_reviews[0].keys())
)

resolved_gap_count=0
gap_gate=False
if gap_structure_ok and GAP_SOURCE:
    source_hash_set=set(gap_source_hashes)
    review_hash_set={r['source_row_sha256'].strip().lower() for r in gap_reviews}
    exact_instance_chain=(review_hash_set==source_hash_set and '' not in review_hash_set)
    if exact_instance_chain:
        for r in gap_reviews:
            day=parse_date(r['review_date_utc'])
            documented=all(str(r[x]).strip() for x in ['reviewer','review_date_utc','independent_review_evidence','resolution_note'])
            decision=r['resolution_status'].strip().upper()
            if documented and day and day<=today and decision in {'APPROVED_RESOLVED','NOT_APPLICABLE_WITH_EVIDENCE'}:
                resolved_gap_count+=1
        gap_gate=(resolved_gap_count==36)

print('Original 36-instance source artifact found:',GAP_SOURCE is not None)
print('Independently resolved gap instances:',resolved_gap_count,'/36')
print('36-gap gate:',gap_gate)


Original 36-instance source artifact found: False
Independently resolved gap instances: 0 /36
36-gap gate: False


In [6]:
# CODE CELL 6/8 — Independent missing-demographics policy review

with DEMO_REVIEW.open(newline='',encoding='utf-8-sig') as f:
    demo=list(csv.DictReader(f))

demo_structure_ok=(
  len(demo)==4
  and {r.get('field','') for r in demo}=={'gender','birthDate','race','ethnicity'}
  and bool(demo)
  and set(DEMO_COLS).issubset(demo[0].keys())
)

demo_public=[]
approved_demo=set()
if demo_structure_ok:
    for r in demo:
        day=parse_date(r['review_date_utc'])
        documented=all(str(r[x]).strip() for x in [
          'proposed_policy','reviewer','review_date_utc','independent_review_evidence'
        ])
        policy_approved=(
          r['verified_source_state'].strip()=='ABSENT_IN_VERIFIED_SYNTHETIC_SOURCE'
          and r['decision'].strip().upper()=='APPROVED'
          and documented and day is not None and day<=today
        )
        if policy_approved: approved_demo.add(r['field'])
        demo_public.append({
          'field':r['field'],
          'verified_source_state':r['verified_source_state'],
          'independent_policy_review_status':'APPROVED' if policy_approved else 'PENDING_OR_INCOMPLETE'
        })

demographic_gate={'gender','birthDate'} <= approved_demo

print('Independent policy approvals:',sorted(approved_demo))
print('Gender/birthDate policy gate:',demographic_gate)


Independent policy approvals: []
Gender/birthDate policy gate: False


In [7]:
# CODE CELL 7/8 — Recalculate the ten Arm A gates without inventing evidence

old={r['gate']:r for r in gates44}
required_gate_names=set(old)
assert required_gate_names=={
 'authorized_frozen_athena_with_valid_manifest',
 '19_binding_human_adjudication',
 'original_36_vocabulary_gap_instances_resolved',
 'source_supported_clinical_procedures',
 'real_dicom_uid_source_attested',
 'missing_gender_and_birthdate_policy_independently_approved',
 'official_MICDM_load_executed',
 'original_unmodified_generic_TFL_armC_executed',
 'externally_independent_challenge_authored',
 'real_authorized_MRI_processed'
}

gate_values={
 'authorized_frozen_athena_with_valid_manifest':manifest_valid,
 '19_binding_human_adjudication':binding_gate,
 'original_36_vocabulary_gap_instances_resolved':gap_gate,
 'source_supported_clinical_procedures':False,
 'real_dicom_uid_source_attested':False,
 'missing_gender_and_birthdate_policy_independently_approved':demographic_gate,
 'official_MICDM_load_executed':False,
 'original_unmodified_generic_TFL_armC_executed':False,
 'externally_independent_challenge_authored':False,
 'real_authorized_MRI_processed':False
}

next_actions={
 'authorized_frozen_athena_with_valid_manifest':'Place authorized CONCEPT.csv/.gz and a SHA-matching SNAPSHOT_MANIFEST.json in the private Athena workspace',
 '19_binding_human_adjudication':'Complete independent evidence review for all 19 binding keys using the frozen snapshot',
 'original_36_vocabulary_gap_instances_resolved':'Locate the original 36-row Phase29 gap artifact and independently resolve all 36 exact row hashes',
 'source_supported_clinical_procedures':'Requires source-attested clinical Procedure evidence; manufactured synthetic fixtures cannot clear this gate',
 'real_dicom_uid_source_attested':'Requires source-attested real study/series DICOM UIDs; synthetic identifiers cannot clear this gate',
 'missing_gender_and_birthdate_policy_independently_approved':'Independently approve the explicit policy for absent gender and birthDate without inventing values',
 'official_MICDM_load_executed':'Execute only after mandatory source and vocabulary gates support the pinned target fields',
 'original_unmodified_generic_TFL_armC_executed':'Run the frozen original generic TFL v15 comparator separately without retrofitting MRI-specific rules into it',
 'externally_independent_challenge_authored':'Obtain independently authored/blinded challenge cases',
 'real_authorized_MRI_processed':'Run only on authorized real MRI records under the frozen protocol'
}

gates45=[]
for g in gates44:
    name=g['gate']
    gates45.append({
      'gate':name,
      'phase44_official_validation_passed':g['official_validation_passed'],
      'phase45_evidence_supported':bool(gate_values[name]),
      'official_validation_passed':bool(gate_values[name]),
      'next_action':next_actions[name]
    })

cleared=[r['gate'] for r in gates45 if r['official_validation_passed']]
remaining=[r['gate'] for r in gates45 if not r['official_validation_passed']]

# Required schema fields remain blockers independent of vocabulary progress.
assert summary44['source_clinical_procedures_attested']==0
assert summary44['real_dicom_uid_source_attested']==0
assert not gate_values['source_supported_clinical_procedures']
assert not gate_values['real_dicom_uid_source_attested']
assert not gate_values['official_MICDM_load_executed']

print('Evidence-supported gates now cleared:',len(cleared),'/10')
for g in cleared: print(' PASS',g)
print('Remaining:',len(remaining))


Evidence-supported gates now cleared: 0 /10
Remaining: 10


In [8]:
# CODE CELL 8/8 — Public-safe Phase 45 outputs and verified Drive save

receipt=[
 {'evidence':'authorized_frozen_athena_snapshot','status':'PRESENT' if ATHENA else 'NOT_FOUND','public_detail':'content and filename kept private'},
 {'evidence':'snapshot_manifest','status':'VALID' if manifest_valid else ('PRESENT_INVALID' if MANIFEST.is_file() else 'NOT_FOUND'),'public_detail':'source/release/hash values not published'},
 {'evidence':'19_binding_review_worksheet','status':'VALID_STRUCTURE' if binding_structure_valid else 'INVALID_STRUCTURE','public_detail':'selected concept IDs and reviewer identities excluded'},
 {'evidence':'original_36_gap_instance_artifact','status':'FOUND_36_ROWS' if GAP_SOURCE else 'NOT_FOUND_AS_36_INSTANCE_FILE','public_detail':'public prior artifact only'},
 {'evidence':'36_gap_independent_review','status':'COMPLETE' if gap_gate else 'INCOMPLETE','public_detail':f'{resolved_gap_count}/36 independently resolved'},
 {'evidence':'demographic_policy_review','status':'GENDER_BIRTHDATE_APPROVED' if demographic_gate else 'INCOMPLETE','public_detail':f'{len(approved_demo)}/4 tracked policies independently approved'},
]

binding_public=[]
for r in status:
    binding_public.append({
      'binding_key':r['binding_key'],
      'status':r['status'],
      'expected_domain_if_defined':r['expected_domain_if_defined'],
      'candidate_present':r['candidate_present'],
      'manual_review_supplied':r['manual_review_supplied'],
      'snapshot_present':r['snapshot_present'],
      'manifest_valid':r['manifest_valid']
    })

def save(name,rows,fields=None):
    p=STAGE/name
    write_csv(p,rows,fields)
    return p

artifacts=[]
artifacts.append(save('ICHI2027_Phase45_PRIVATE_EVIDENCE_RECEIPT_STATUS.csv',receipt))
artifacts.append(save('ICHI2027_Phase45_BINDING_REVIEW_STATUS_PUBLIC.csv',binding_public))
artifacts.append(save('ICHI2027_Phase45_DEMOGRAPHIC_POLICY_STATUS_PUBLIC.csv',demo_public if demo_public else [
 {'field':x,'verified_source_state':'ABSENT_IN_VERIFIED_SYNTHETIC_SOURCE','independent_policy_review_status':'PENDING_OR_INCOMPLETE'}
 for x in ['gender','birthDate','race','ethnicity']
]))
artifacts.append(save('ICHI2027_Phase45_ARM_A_GATES.csv',gates45))
artifacts.append(save('ICHI2027_Phase45_LOGICAL_CONTRACT_CARRIED_FORWARD.csv',contract44))

readme=STAGE/'ICHI2027_Phase45_README.txt'
readme.write_text(
 'Phase 45 revalidates private Athena/binding evidence against the frozen Phase 44 standards chain.\n'
 'Public outputs contain no selected concept IDs, Athena records, reviewer identities, or private paths.\n'
 'The 19-binding gate requires strict frozen-snapshot and independent-review evidence.\n'
 'The 36-gap gate requires an exact 36-row original gap-instance chain plus independent row-level resolution.\n'
 'Passing vocabulary or demographic-policy gates does not clear the source Procedure or DICOM UID gates.\n'
 'Official MI-CDM loading remains false until the required source evidence and all mandatory mappings are supported.\n',
 encoding='utf-8'
)
artifacts.append(readme)

summary={
 'project':PROJECT,
 'phase':PHASE,
 'run_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'privacy':'PUBLIC_METADATA_ONLY_NO_CONCEPT_IDS_NO_REVIEWER_IDENTITIES_NO_PRIVATE_PATHS',
 'input_phase44_shareable_sha256':sha256(P44['summary']),
 'phase44_omop_version':summary44['omop_version'],
 'phase44_omop_commit':summary44['omop_repo_commit'],
 'phase44_micdm_declared_ddl_version':summary44['micdm_declared_ddl_version'],
 'phase44_imagewg_commit':summary44['imagewg_repo_commit'],
 'private_gate':{
   'snapshot_supplied':ATHENA is not None,
   'snapshot_manifest_valid':manifest_valid,
   'athena_schema_valid':athena_schema_valid if ATHENA else False,
   'manual_snapshot_checked_approvals':len(approved_bindings),
   'reviewed_inapplicable_gender_keys':len(inapplicable),
   'binding_keys_total':19,
   'binding_gate_complete':binding_gate,
   'original_36_gap_source_instances_found':36 if GAP_SOURCE else 0,
   'independently_resolved_gap_instances':resolved_gap_count,
   'original_36_vocabulary_gaps_declared_resolved':gap_gate,
   'approved_demographic_policy_fields_count':len(approved_demo),
   'missing_gender_and_birthdate_policy_independently_approved':demographic_gate
 },
 'armA':{
   'evidence_supported_gates':len(cleared),
   'remaining_gates':len(remaining),
   'cleared_gate_names':cleared,
   'source_clinical_procedures_attested':False,
   'real_dicom_uid_source_attested':False,
   'official_micdm_load_executed':False,
   'official_omop_etl_executed':False,
   'status':'BLOCKED_OFFICIAL_ARM_A_NOT_EXECUTED'
 },
 'frozen_original_generic_tfl_armC_executed':False,
 'externally_independent_challenge_authored':False,
 'real_authorized_mri_processed':False,
 'full_A_B_C_D_evaluation':False,
 'artifact_sha256':{p.name:sha256(p) for p in artifacts},
 'warning':'Private evidence gates may advance only when exact authorized and independently reviewed evidence is present; no missing source evidence is synthesized.'
}
summary_path=STAGE/'ICHI2027_Phase45_SHAREABLE.json'
summary_path.write_text(json.dumps(summary,sort_keys=True,indent=2)+'\n',encoding='utf-8')
artifacts.append(summary_path)

zip_path=STAGE/'ICHI2027_Phase45_PUBLIC_OUTPUTS.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in artifacts:z.write(p,p.name)

OUT.mkdir(parents=True,exist_ok=False)
for p in artifacts+[zip_path]:
    dest=OUT/p.name
    for attempt in range(3):
        try:
            shutil.copyfile(p,dest)
            assert sha256(dest)==sha256(p)
            break
        except Exception as e:
            if attempt==2: raise RuntimeError('Drive save/readback verification failed for '+p.name) from e
            time.sleep(1+attempt)

print('SUCCESS: Phase 45 evidence revalidation completed.')
print('Evidence-supported gates:',len(cleared),'/10')
print('Remaining gates:',len(remaining))
print('Shareable output:',OUT/summary_path.name)
print('Public output ZIP:',OUT/zip_path.name)
print('Private review files remain only in the private evidence workspace.')


SUCCESS: Phase 45 evidence revalidation completed.
Evidence-supported gates: 0 /10
Remaining gates: 10
Shareable output: /content/drive/MyDrive/ICHI2027/Phase45/run_20261001T020553_159729Z/ICHI2027_Phase45_SHAREABLE.json
Public output ZIP: /content/drive/MyDrive/ICHI2027/Phase45/run_20261001T020553_159729Z/ICHI2027_Phase45_PUBLIC_OUTPUTS.zip
Private review files remain only in the private evidence workspace.


## What to use from this phase

The important output is `ICHI2027_Phase45_SHAREABLE.json`.

If the Athena snapshot or independent reviews are still missing, Phase 45 is still useful: it creates the exact private review templates and shows which evidence is missing without weakening the gate.

If some gates clear, the next phase should use only those verified approvals. It must still respect the Phase 44 finding that source-attested clinical Procedure and real DICOM UID evidence are currently absent.
